In [1]:
import pickle
import os
import pandas as pd
from collections import Counter, defaultdict

In [2]:
# pred_tuples_llm_dict = pickle.load(open('composition_test_results_claude.pkl','rb'))
gold_disco_tuples_test = pickle.load(open('all_gold_tuples_test.pkl', 'rb'))
print(gold_disco_tuples_test[:10])

[('S0022309304006957_0_2_1_0_1', 'PbO', 30.0, 'mol'), ('S0022309304006957_0_2_1_0_1', 'SiO2', 70.0, 'mol'), ('S0022309304006957_0_2_2_0_2', 'PbO', 33.0, 'mol'), ('S0022309304006957_0_2_2_0_2', 'SiO2', 67.0, 'mol'), ('S0022309304006957_0_2_3_0_3', 'PbO', 43.0, 'mol'), ('S0022309304006957_0_2_3_0_3', 'SiO2', 57.0, 'mol'), ('S0022309304006957_0_2_4_0_4', 'PbO', 50.0, 'mol'), ('S0022309304006957_0_2_4_0_4', 'SiO2', 50.0, 'mol'), ('S0022309304003618_0_1_1_0', 'Si', 0.336, 'mol'), ('S0022309304003618_0_1_1_0', 'Ca', 0.086, 'mol')]


In [3]:
# pred_tuples_llm_dict
pred_tuples_llm_dict_org = pickle.load(open('composition_test_results_deepseek_chat_v3.pkl','rb'))
print(len(pred_tuples_llm_dict_org.keys()))
print(type(pred_tuples_llm_dict_org))
# pred_tuples_llm_dict_org = pickle.load(open('composition_test_results_gemini_with_abstract_without_truncation_explicit.pkl','rb'))

737
<class 'dict'>


In [3]:
# pred_tuples_llm_dict_org[('S0022309300000570', 0)]

In [2]:
# print(pred_tuples_llm_dict_org)

In [6]:
pred_tuples_llm_dict = {}

for key, value in pred_tuples_llm_dict_org.items():
    # Remove "Table-Type = X.\n" if it exists
    cleaned_value = value.split("\n")[-1].strip()

    # If starts with '[[' and ends with ']', but not ']]', add ']' at the end
    if cleaned_value.startswith('[[') and cleaned_value.endswith(']') and not cleaned_value.endswith(']]'):
        cleaned_value = cleaned_value + "]"
    
    # If starts with '[' but not '[[' and ends with ']]', add '[' to the beginning
    elif cleaned_value.startswith('[') and not cleaned_value.startswith('[[') and cleaned_value.endswith(']]'):
        cleaned_value = "[" + cleaned_value

    # Ensure empty tables remain as '[]'
    if cleaned_value.strip() == '':
        cleaned_value = '[]'

    
    pred_tuples_llm_dict[key] = cleaned_value

In [7]:
def process_dict_to_list_of_tuples(data_dict):
    result = []
    
    for value in data_dict.values():
        # Skip empty lists
        # print(value)
        # print()
        if (value == '[]' or value == ']' or 'Internal error' in value or 'Error:' in value or value == '</html>'):
            continue

        if '...' in value:
            print(f'... in value = {value}')
            continue
            
            
        # Convert string representation to actual list
        # Remove outer quotes and evaluate
        # print(value)
        list_data = eval(value)
        
        # Flatten the list of lists and extend result
        for sublist in list_data:
            for item in sublist:
                if type(item[2])!=str: #llms sometimes return values like 'traces', ('S0022309304008944_2_2_3_0', 'Gahnite', 'traces', 'wt'), only 1 was present
                    result.append(item)
    
    return result

In [8]:
pred_disco_tuples_test = process_dict_to_list_of_tuples(pred_tuples_llm_dict)
print(len(gold_disco_tuples_test), len(pred_disco_tuples_test))

9540 9199


In [9]:
def sort_comp(single_list_comp):
    '''
    for sorting  ordering of chemical components alphabetically
    '''
    single_list_comp_sorted = sorted(single_list_comp, key=lambda x: x[0])
    return single_list_comp_sorted

def get_composition_metrics_without_ids(gold_tuples, pred_tuples):
    gold_comps, pred_comps = defaultdict(set), defaultdict(set)
    
    for g in gold_tuples:
        underscore_count = g[0].count('_')
        if underscore_count == 5:
            new_item = '_'.join(g[0].split('_')[:-1])
        else:
            new_item = g[0]
        gold_comps[new_item].add((g[1], round(g[2], 2), g[3]))
#         print(gold_comps[g[0]])

    for p in pred_tuples:
        underscore_count = p[0].count('_')
        if underscore_count == 5:
            new_item = '_'.join(p[0].split('_')[:-1])
        else:
            new_item = p[0]
        pred_comps[new_item].add((p[1], round(p[2], 2), p[3]))
        
   
    for p, v in pred_comps.items():
        pred_comps[p] = set(sort_comp(list(v)))
        
    for g, v in gold_comps.items():
        gold_comps[g] = set(sort_comp(list(v)))

    prec = 0
    for p, v in pred_comps.items():
        if p in gold_comps and gold_comps[p] == v:
            prec += 1
    if len(pred_comps) > 0:
        prec /= len(pred_comps)
    else:
        prec = 0.0
    rec = 0
    for g, v in gold_comps.items():
        if g in pred_comps and pred_comps[g] == v:
            rec += 1
    rec /= len(gold_comps)
    fscore = 2 * prec * rec / (prec + rec) if (prec + rec > 0) else 0.0
    metrics = {'precision': prec, 'recall': rec, 'fscore': fscore}
    metrics = {m: round(v * 100, 2) for m, v in metrics.items()}
    return metrics

In [10]:
def normalize_compounds(tuples_list):
    """
    Normalize compound scores to a 0-100 scale if the sum of compounds for a material is ≤1.
    
    Args:
        tuples_list (list): List of tuples where each tuple is (id, compound, value, unit)
    
    Returns:
        list: Normalized list of tuples
    """
    # Group tuples by ID
    grouped_data = defaultdict(list)
    for tup in tuples_list:
        grouped_data[tup[0]].append(tup)
    
    # Process each group and normalize if needed
    normalized_tuples = []
    for material_id, compounds in grouped_data.items():
        # Calculate sum of values for this material
        total_value = sum(comp[2] for comp in compounds)
        
        # If sum ≤1, multiply each value by 100
        multiplier = 100 if total_value <=1.05 else 1
        
        # Create new normalized tuples
        for comp in compounds:
            normalized_tuple = (
                comp[0],  # ID
                comp[1],  # compound
                round(comp[2] * multiplier, 1),  # normalized value
                comp[3]   # unit
            )
            normalized_tuples.append(normalized_tuple)
    
    return normalized_tuples

In [11]:
# pred_disco_tuples_test = pickle.load(open('all_pred_tuples_test.pkl', 'rb'))
gold_disco_tuples_test_round = normalize_compounds(gold_disco_tuples_test)
pred_disco_tuples_test_round = normalize_compounds(pred_disco_tuples_test)
### This round off has been considered to be fair to the baselines, which increases baseline score significantly

In [14]:
print(get_composition_metrics_without_ids(gold_disco_tuples_test_round, pred_disco_tuples_test_round))

{'precision': 49.71, 'recall': 52.26, 'fscore': 50.95}


In [11]:
# pred_tuples_llm_dict